In [1]:
import sys
sys.path.append("..")

from src.analytical.build_base_analitica import build_base_analitica




In [2]:
import pandas as pd

dim_municipio = pd.read_parquet(
    "../data/gold/dimensions/dim_municipio/dim_municipio.parquet"
)

df = pd.read_parquet(
    "../data/gold/facts/fato_alfabetizacao_municipio/fato_alfabetizacao_municipio.parquet"
)

In [3]:
df_analitica = build_base_analitica(
    df,
    dim_municipio
)

In [4]:
df_analitica.shape

(4611, 11)

In [5]:
df_analitica["target_atingiu_meta_2024"].value_counts(normalize=True)

target_atingiu_meta_2024
1    0.535242
0    0.464758
Name: proportion, dtype: float64

In [6]:
df_analitica.isna().sum()

id_municipio                 0
taxa_alfabetizacao_2023      0
taxa_presenca_2023           0
taxa_preenchimento_2023      0
alunos_avaliados_2023        0
alunos_alfabetizados_2023    0
alunos_presentes_2023        0
provas_preenchidas_2023      0
meta_2024                    0
target_atingiu_meta_2024     0
UF                           0
dtype: int64

### Estatísticas da base analítica

In [7]:
print("===== SHAPE =====")
print(df_analitica.shape)

print("\n===== COLUNAS =====")
print(df_analitica.columns.tolist())

print("\n===== TIPOS =====")
print(df_analitica.dtypes)

print("\n===== NULOS =====")
print(df_analitica.isna().sum())

print("\n===== MUNICÍPIOS =====")
print(df_analitica["id_municipio"].nunique())

print("\n===== UFs =====")
print(df_analitica["UF"].nunique())

===== SHAPE =====
(4611, 11)

===== COLUNAS =====
['id_municipio', 'taxa_alfabetizacao_2023', 'taxa_presenca_2023', 'taxa_preenchimento_2023', 'alunos_avaliados_2023', 'alunos_alfabetizados_2023', 'alunos_presentes_2023', 'provas_preenchidas_2023', 'meta_2024', 'target_atingiu_meta_2024', 'UF']

===== TIPOS =====
id_municipio                     str
taxa_alfabetizacao_2023      float64
taxa_presenca_2023           float64
taxa_preenchimento_2023      float64
alunos_avaliados_2023          int64
alunos_alfabetizados_2023      int64
alunos_presentes_2023          int64
provas_preenchidas_2023        int64
meta_2024                    float64
target_atingiu_meta_2024       int64
UF                               str
dtype: object

===== NULOS =====
id_municipio                 0
taxa_alfabetizacao_2023      0
taxa_presenca_2023           0
taxa_preenchimento_2023      0
alunos_avaliados_2023        0
alunos_alfabetizados_2023    0
alunos_presentes_2023        0
provas_preenchidas_2023     

In [8]:
df_analitica["target_atingiu_meta_2024"].value_counts(
    normalize=True
).mul(100).round(2)

target_atingiu_meta_2024
1    53.52
0    46.48
Name: proportion, dtype: float64

In [9]:
df_analitica.groupby(
    "target_atingiu_meta_2024"
).agg(
    municipios=("id_municipio", "count"),
    alfabetizacao_2023=("taxa_alfabetizacao_2023", "mean"),
    presenca_2023=("taxa_presenca_2023", "mean"),
    preenchimento_2023=("taxa_preenchimento_2023", "mean"),
    alunos_avaliados=("alunos_avaliados_2023", "mean")
)

,municipios,alfabetizacao_2023,presenca_2023,preenchimento_2023,alunos_avaliados
target_atingiu_meta_2024,,,,,
0,2143,60.597298,88.455282,88.448502,381.081661
1,2468,60.374968,90.961592,90.952715,290.008104


In [10]:
df_analitica.groupby("UF").agg(
    municipios=("id_municipio", "count"),
    taxa_alfabetizacao_2023=(
        "taxa_alfabetizacao_2023",
        "mean"
    ),
    taxa_presenca_2023=(
        "taxa_presenca_2023",
        "mean"
    ),
    taxa_preenchimento_2023=(
        "taxa_preenchimento_2023",
        "mean"
    ),
    percentual_atingiu_meta=(
        "target_atingiu_meta_2024",
        "mean"
    )
).sort_values(
    "municipios",
    ascending=False
)

,municipios,taxa_alfabetizacao_2023,taxa_presenca_2023,taxa_preenchimento_2023,percentual_atingiu_meta
UF,,,,,
MG,801,62.854120,91.944906,91.944906,0.799001
RS,421,73.529739,87.552660,87.552660,0.097387
PR,395,75.937316,89.254177,89.254177,0.470886
BA,394,37.802157,86.792234,86.792234,0.124365
GO,242,73.042479,89.672934,89.672934,0.805785
SC,238,67.637521,84.900714,84.900714,0.630252
PI,224,60.109063,95.537009,95.537009,0.584821
PB,220,54.201409,90.749273,90.749273,0.513636
MA,216,61.160463,91.182500,91.182500,0.523148


Observa-se que na base atual existem estados, como o caso de SP, que estão sub-representados em volume de municípios. Todavia, sabemos que existe bases auxiliares que podem ser usados para enriquecer a base de dados. 